<a href="https://colab.research.google.com/github/Prajwaln08/SQLSmith/blob/main/notebooks/09_QLoRA_SFT_7B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQLSmith — 09 | QLoRA SFT: Qwen2.5-Coder-7B-Instruct

**Milestone B · Google Colab T4 (16 GB) · Spider 1.0 training-only supervision**

Fine-tunes the **7B** model using the *same frozen Notebook 07* database-disjoint split used for the 1.5B model. Training uses **4-bit NF4**, LoRA adapters, assistant-answer-only loss, validation-only checkpoint monitoring, and separate results/checkpoints.

**Important:** This is not an official execution-accuracy evaluation. Notebook 10 will load the adapter and evaluate on the *reserved* Spider development set (1,034 examples). Current frozen 7B baseline: **73.2%**.

**T4 safeguards:** maximum sequence length 1024, LoRA `r=8`, micro-batch 1, gradient accumulation 16, gradient checkpointing, 4-bit quantization. These differ intentionally from Notebook 08; record them when interpreting performance. A smoke test uses a *different run directory* so that its checkpoint cannot contaminate the main experiment.

**Execution:** run cells in order, CPU→GPU is **not** needed (use a T4 for the whole notebook). Restart after package installation if the runtime asks. Inspect Section 12 outputs before committing to GitHub; notebook output may contain local paths and metrics.

## 1 — Install packages

Use a **Transformers 4.x** range to avoid the API migration encountered in Notebook 08. The notebook additionally introspects `TrainingArguments` to handle evaluation argument naming. Restart the Colab session **once** after this cell if packages were changed; then run from Section 2 onward.

In [1]:
%pip install -q -U "transformers>=4.56,<5" "accelerate>=1.10,<2" "peft>=0.17,<1" "bitsandbytes>=0.46.1" "datasets>=3,<5" "safetensors>=0.4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2 — Project paths, GPU checks, and experiment settings

`RUN_NAME='smoke_10'` and `MAX_STEPS=10` give a small initial wiring test. For the main experiment, change them **together** to `RUN_NAME='main_200'` and `MAX_STEPS=200`, then restart the runtime and rerun from Section 2. Do not reuse the smoke checkpoint with a different schedule.

**No baseline predictions or Notebook 08 files are overwritten.**

In [3]:
from google.colab import drive
from pathlib import Path
import os, json, random, time, hashlib, inspect, importlib.metadata, platform, gc
from datetime import datetime, timezone
import numpy as np
import torch

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Prajwal/Projects/SQLSmith')
DATA_DIR = PROJECT / 'data/processed/sft_v1'
MODEL_ROOT = PROJECT / 'models/qlora_7b_sft_v1'
REPORT_DIR = PROJECT / 'results/milestone_B'
TRAIN_FILE = DATA_DIR / 'train_chat.jsonl'
VAL_FILE = DATA_DIR / 'val_chat.jsonl'
MANIFEST = DATA_DIR / 'dataset_manifest.json'
for p in (TRAIN_FILE, VAL_FILE, MANIFEST):
    assert p.is_file(), f'Missing Notebook 07 artifact: {p}'

MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
SEED = 42
MAX_SEQ_LEN = 1024
VAL_EXAMPLES = 64  # Selected consistently across database IDs
LR = 1e-4
RUN_NAME = 'smoke_10'     # For main run: 'main_200'
MAX_STEPS = 10           # For main run: 200
SAVE_STEPS = 10          # Main run: keep 25 (or 10), with matching RUN_NAME
EVAL_STEPS = 10

assert RUN_NAME in {'smoke_10', 'main_200'}, 'Use a documented run name'
if RUN_NAME == 'smoke_10': assert MAX_STEPS == 10
if RUN_NAME == 'main_200': assert MAX_STEPS == 200
OUT_DIR = MODEL_ROOT / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU'
GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
assert VRAM >= 14, f'This configuration expects a ~16GB GPU; found {VRAM:.1f} GiB'
print('GPU:', GPU, '| available VRAM:', round(VRAM, 2), 'GiB')
print('Run:', RUN_NAME, '| steps:', MAX_STEPS, '| output:', OUT_DIR)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def sha256_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
print('Training SHA:', sha256_file(TRAIN_FILE))
print('Validation SHA:', sha256_file(VAL_FILE))

Mounted at /content/drive
GPU: Tesla T4 | available VRAM: 14.56 GiB
Run: smoke_10 | steps: 10 | output: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/models/qlora_7b_sft_v1/smoke_10
Training SHA: 8d616419aebed7ddaf55f1ae02331ea041c76f34b732219ed7ba37493cc504ba
Validation SHA: 474a0680030b288d8d66b2aafae5b93d2ddd6c3793a1dbc59632703772c7a9ab


## 3 — Load and verify the exact Notebook 07 split

Do not use Spider dev here: the official benchmark must stay reserved for final comparison.

In [4]:
def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
train_rows = read_jsonl(TRAIN_FILE)
val_rows = read_jsonl(VAL_FILE)
assert (len(train_rows), len(val_rows)) == (7931, 717), (len(train_rows), len(val_rows))
assert all(len(x['messages'])==3 and x['messages'][-1]['role']=='assistant' for x in train_rows+val_rows)
train_db = {x['db_id'] for x in train_rows}
val_db = {x['db_id'] for x in val_rows}
assert (len(train_db),len(val_db)) == (131,15)
assert train_db.isdisjoint(val_db), 'TRAIN / VALIDATION DATABASE LEAKAGE'
assert {x['id'] for x in train_rows}.isdisjoint({x['id'] for x in val_rows})
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
print('Training',len(train_rows),'from',len(train_db),'DBs')
print('Validation',len(val_rows),'from',len(val_db),'DBs')
print('Manifest keys:', list(manifest)[:12])
print('Sample role order:', [m['role'] for m in train_rows[0]['messages']])

Training 7931 from 131 DBs
Validation 717 from 15 DBs
Manifest keys: ['project', 'stage', 'dataset', 'source_repository', 'source_archive_sha256', 'sources', 'seed', 'validation_fraction_of_database_ids', 'train_database_ids', 'validation_database_ids', 'train_examples', 'validation_examples']
Sample role order: ['system', 'user', 'assistant']


## 4 — Fixed database-stratified validation sample

Internal validation is for loss monitoring, not the official test-suite accuracy. Choose at least one record from each of the 15 validation databases, then use seeded sampling across the remainder.

In [5]:
from collections import defaultdict
by_db=defaultdict(list)
for row in val_rows: by_db[row['db_id']].append(row)
for rows in by_db.values(): rows.sort(key=lambda r:r['id'])
assert VAL_EXAMPLES >= len(by_db)
chosen=[by_db[db][0] for db in sorted(by_db)]
rest=[r for db in sorted(by_db) for r in by_db[db][1:]]
rng=random.Random(SEED); rng.shuffle(rest)
chosen += rest[:VAL_EXAMPLES-len(chosen)]
assert len(chosen)==VAL_EXAMPLES
val_sample_ids=[r['id'] for r in chosen]
print('Validation selected:', len(chosen), '| databases:', len({x['db_id'] for x in chosen}))

Validation selected: 64 | databases: 15


## 5 — Verify library versions and load Qwen 7B in 4-bit NF4

The HF Hub's unauthenticated-request warning is harmless for this public model. An `ImportError` mentioning `bitsandbytes` means restart Colab after package installation and rerun setup. Avoid simultaneously loading the 1.5B model into GPU memory.

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

for p in ('torch','transformers','peft','bitsandbytes','accelerate'):
    print(p, importlib.metadata.version(p))

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map={'': 0}, dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Allocated GPU GiB:', round(torch.cuda.memory_allocated()/2**30, 2))
assert torch.cuda.memory_allocated()/2**30 < VRAM-1.0, 'Insufficient GPU headroom after model loading'

/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:52: FutureWarning: Support for Transformers < 5.6 is deprecated in PEFT and will be removed in the next release, v1.0. You have Transformers 4.57.6. Please upgrade to Transformers >= 5.6.
  warnings.warn(


torch 2.14.1+cu130
transformers 4.57.6
peft 0.22.0
bitsandbytes 0.50.2
accelerate 1.16.0


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (7 outputs left out)

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643
Allocated GPU GiB: 7.28


## 6 — Tokenize with SQL answer-only labels

The model must learn **only assistant SQL tokens**. All prompt tokens are `-100` in labels, and long examples are excluded (not silently truncated) so SQL targets remain intact. This keeps the objective comparable to Notebook 08 while lowering the context cap for the 7B T4 run.

In [7]:
from datasets import Dataset

def tokenize_row(row):
    msgs=row['messages']
    prompt=tokenizer.apply_chat_template(msgs[:-1],tokenize=False,add_generation_prompt=True)
    answer=msgs[-1]['content']
    prompt_ids=tokenizer(prompt,add_special_tokens=False)['input_ids']
    response_ids=tokenizer(answer,add_special_tokens=False)['input_ids']
    if tokenizer.eos_token_id is not None:
        response_ids=response_ids+[tokenizer.eos_token_id]
    ids=prompt_ids+response_ids
    if not response_ids or len(ids)>MAX_SEQ_LEN: return None
    return {'input_ids':ids,'attention_mask':[1]*len(ids),
            'labels':[-100]*len(prompt_ids)+response_ids,
            'id':row['id'],'db_id':row['db_id'],'length':len(ids),
            'target_length':len(response_ids)}

train_encoded_raw=[tokenize_row(r) for r in train_rows]
val_encoded_raw=[tokenize_row(r) for r in chosen]
train_dropped=sum(x is None for x in train_encoded_raw)
val_dropped=sum(x is None for x in val_encoded_raw)
train_encoded=[x for x in train_encoded_raw if x is not None]
val_encoded=[x for x in val_encoded_raw if x is not None]
assert train_encoded and val_encoded
assert all(any(x!=-100 for x in row['labels']) for row in train_encoded+val_encoded)
print('Training kept:',len(train_encoded),'| excluded >',MAX_SEQ_LEN,'tokens:',train_dropped)
print('Validation kept:',len(val_encoded),'| excluded >',MAX_SEQ_LEN,'tokens:',val_dropped)
print('Validation DB coverage after length filtering:',len({x['db_id'] for x in val_encoded}),'of',len(val_db))
assert len({x['db_id'] for x in val_encoded})>=12, 'Too many validation DBs excluded; review context length'

train_dataset=Dataset.from_list([
    {k:v for k,v in row.items() if k in ('input_ids','attention_mask','labels')}
    for row in train_encoded])
eval_dataset=Dataset.from_list([
    {k:v for k,v in row.items() if k in ('input_ids','attention_mask','labels')}
    for row in val_encoded])

Training kept: 7849 | excluded > 1024 tokens: 82
Validation kept: 58 | excluded > 1024 tokens: 6
Validation DB coverage after length filtering: 14 of 15


## 7 — Assistant-only dynamic collator and safety checks

**Do not use** `DataCollatorForLanguageModeling` here; it would replace the assistant-only loss labels.

In [8]:
from dataclasses import dataclass

@dataclass
class AssistantOnlyCollator:
    pad_id: int
    pad_to_multiple_of: int = 8
    def __call__(self, features):
        length=max(len(f['input_ids']) for f in features)
        length=((length+self.pad_to_multiple_of-1)//self.pad_to_multiple_of)*self.pad_to_multiple_of
        out={'input_ids':[],'attention_mask':[],'labels':[]}
        for f in features:
            pad=length-len(f['input_ids'])
            out['input_ids'].append(f['input_ids']+[self.pad_id]*pad)
            out['attention_mask'].append(f['attention_mask']+[0]*pad)
            out['labels'].append(f['labels']+[-100]*pad)
        return {k:torch.tensor(v,dtype=torch.long) for k,v in out.items()}

collator=AssistantOnlyCollator(tokenizer.pad_token_id)
batch=collator([train_dataset[0],train_dataset[1]])
assert batch['input_ids'].shape==batch['labels'].shape
assert torch.all(batch['labels'][batch['attention_mask']==0]==-100)
assert int((batch['labels']!=-100).sum())>0
print('Test batch shape:',tuple(batch['input_ids'].shape))
print('Answer tokens in test batch:',int((batch['labels']!=-100).sum()))

Test batch shape: (2, 184)
Answer tokens in test batch: 29


## 8 — Training arguments and Trainer (compatible with 4.x/5.x naming)

Avoid the removed `warmup_ratio` argument that failed in Notebook 08: compute `warmup_steps` explicitly. Select `eval_strategy`/`evaluation_strategy` only if supported by the installed library.

**Training configuration:** micro-batch 1; accumulation 16; cosine LR with 3% warmup; 8-bit paged AdamW; fp16; checkpoint every `SAVE_STEPS`. Running on Drive keeps checkpoints persistent but may reduce I/O speed.

In [9]:
from transformers import Trainer, TrainingArguments

supported=inspect.signature(TrainingArguments).parameters
kwargs=dict(
    output_dir=str(OUT_DIR/'trainer_checkpoints'),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=max(1,round(MAX_STEPS*0.03)),
    weight_decay=0.0,
    optim='paged_adamw_8bit',
    fp16=True,
    bf16=False,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=5,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    max_grad_norm=0.3,
    group_by_length=True,
)
if 'eval_strategy' in supported:
    kwargs['eval_strategy']='steps'
elif 'evaluation_strategy' in supported:
    kwargs['evaluation_strategy']='steps'
else:
    raise RuntimeError('Installed TrainingArguments has no supported evaluation strategy argument')
kwargs['eval_steps']=EVAL_STEPS
if 'gradient_checkpointing' in supported:
    kwargs['gradient_checkpointing']=True
elif 'activation_checkpointing' in supported:
    kwargs['activation_checkpointing']=True
else:
    print('Trainer checkpoint option absent; model is already prepared for checkpointing')
unsupported=set(kwargs)-set(supported)
assert not unsupported, f'Unsupported TrainingArguments fields: {sorted(unsupported)}'
training_args=TrainingArguments(**kwargs)
trainer=Trainer(model=model,args=training_args,
               train_dataset=train_dataset,eval_dataset=eval_dataset,
               data_collator=collator)
print('Train args ready:',RUN_NAME,'| steps:',MAX_STEPS,'| warmup:',kwargs['warmup_steps'])
print('Checkpoints:',training_args.output_dir)

Train args ready: smoke_10 | steps: 10 | warmup: 1
Checkpoints: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/models/qlora_7b_sft_v1/smoke_10/trainer_checkpoints


## 9 — Train or resume (safe separate run directories)

Checkpoints are resumable **only with the same `RUN_NAME`, steps, seed, dataset and hyperparameters**. If you changed configuration, use a new run name and output folder instead of resuming a mismatched optimizer schedule.

If CUDA reports out-of-memory during training, restart runtime and reduce `MAX_SEQ_LEN` to 768, using a **new run directory**; don't overwrite a partially completed run with a different sequence cap.

In [11]:

# Fix QLoRA gradient checkpointing on T4
# Run after Section 8, before trainer.train()

import torch

# Disable the previous checkpointing configuration
model.gradient_checkpointing_disable()

# Enable non-reentrant checkpointing
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Keep Trainer from re-enabling the old configuration
trainer.args.gradient_checkpointing = False

# Verify LoRA parameters are trainable
trainable = [
    (name, p) for name, p in model.named_parameters()
    if p.requires_grad
]

print("Trainable parameter tensors:", len(trainable))
assert trainable, "No trainable LoRA parameters found"

# Check the backward graph with a single batch
batch = collator([train_dataset[0]])
batch = {
    k: v.to(model.device)
    for k, v in batch.items()
}

model.train()
outputs = model(**batch)

print("Loss:", outputs.loss.item())
print("Loss requires grad:", outputs.loss.requires_grad)

assert outputs.loss.requires_grad, (
    "Gradient graph is still disconnected"
)

del outputs, batch
model.zero_grad(set_to_none=True)

print("Gradient checkpointing fix verified.")


Trainable parameter tensors: 392
Loss: 0.42429447174072266
Loss requires grad: True
Gradient checkpointing fix verified.


In [12]:
from transformers.trainer_utils import get_last_checkpoint
checkpoint_root=Path(training_args.output_dir)
checkpoint_root.mkdir(parents=True,exist_ok=True)
last_checkpoint=get_last_checkpoint(str(checkpoint_root))
print('Resume checkpoint:',last_checkpoint or 'None — starting new run')
started=time.perf_counter()
train_result=trainer.train(resume_from_checkpoint=last_checkpoint)
training_seconds=time.perf_counter()-started
print('Training finished; seconds:',round(training_seconds,1))
print('Metrics:',train_result.metrics)
print('Peak allocated GPU GiB:',round(torch.cuda.max_memory_allocated()/2**30,2))

Resume checkpoint: None — starting new run


Step,Training Loss,Validation Loss
10,0.259400,0.169860


Training finished; seconds: 503.8
Metrics: {'train_runtime': 484.2967, 'train_samples_per_second': 0.33, 'train_steps_per_second': 0.021, 'total_flos': 4200025157959680.0, 'train_loss': 0.2848277688026428, 'epoch': 0.02038476239011339}
Peak allocated GPU GiB: 10.23


## 10 — Evaluate internal validation loss and save LoRA adapter

This adapter is **not** a standalone model; later inference will load the base Qwen 7B and then attach the LoRA weights. `trainer.evaluate()` reports held-out SFT loss, not Spider execution accuracy.

In [13]:
validation_metrics=trainer.evaluate()
print('Internal validation metrics:',validation_metrics)
ADAPTER_DIR=OUT_DIR/'final_adapter'
ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR),safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))
assert (ADAPTER_DIR/'adapter_config.json').is_file()
assert any(ADAPTER_DIR.glob('adapter_model.*'))
print('Adapter saved:',ADAPTER_DIR)
print('Adapter files:',sorted(p.name for p in ADAPTER_DIR.iterdir()))

Internal validation metrics: {'eval_loss': 0.16986048221588135, 'eval_runtime': 29.5537, 'eval_samples_per_second': 1.963, 'eval_steps_per_second': 1.963, 'epoch': 0.02038476239011339}
Adapter saved: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/models/qlora_7b_sft_v1/smoke_10/final_adapter
Adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


## 11 — Reproducible training report with checksums

Report includes the exact training data hashes, accepted/excluded record counts, sample IDs, software versions, optimization parameters, adapter path and GPU resource usage. Do not claim a post-SFT execution score here.

In [14]:
versions={}
for p in ('torch','transformers','accelerate','peft','bitsandbytes','datasets','safetensors'):
    try:versions[p]=importlib.metadata.version(p)
    except importlib.metadata.PackageNotFoundError:versions[p]='missing'

def jsonable(v):
    if isinstance(v,(np.floating,np.integer)):return v.item()
    if isinstance(v,torch.Tensor):return v.item() if v.numel()==1 else str(v.shape)
    return v

adapter_hashes={p.name:sha256_file(p) for p in ADAPTER_DIR.iterdir() if p.is_file()}
report={
    'project':'SQLSmith', 'notebook':'09_QLoRA_SFT_7B','milestone':'B',
    'run_name':RUN_NAME,'model_id':MODEL_ID,
    'method':'QLoRA NF4 + LoRA r8; answer-only SQL labels',
    'train_original':len(train_rows),'validation_original':len(val_rows),
    'train_accepted':len(train_encoded),'train_dropped_sequence_length':train_dropped,
    'internal_validation_selected':len(chosen),'internal_validation_accepted':len(val_encoded),
    'internal_validation_dropped_sequence_length':val_dropped,
    'train_databases':len(train_db),'validation_databases':len(val_db),
    'seed':SEED,'max_seq_len':MAX_SEQ_LEN,'val_sample_ids':val_sample_ids,
    'train_sha256':sha256_file(TRAIN_FILE),'validation_sha256':sha256_file(VAL_FILE),
    'train_steps':MAX_STEPS,'train_steps_completed':trainer.state.global_step,
    'learning_rate':LR,'lora_rank':8,'lora_alpha':16,
    'micro_batch_size':1,'gradient_accumulation_steps':16,
    'warmup_steps':kwargs['warmup_steps'],'eval_steps':EVAL_STEPS,'save_steps':SAVE_STEPS,
    'adapter_dir':str(ADAPTER_DIR),'adapter_file_hashes':adapter_hashes,
    'training_metrics':{k:jsonable(v) for k,v in train_result.metrics.items()},
    'validation_metrics':{k:jsonable(v) for k,v in validation_metrics.items()},
    'training_seconds':round(training_seconds,2),
    'gpu':GPU,'peak_gpu_memory_allocated_gib':round(torch.cuda.max_memory_allocated()/2**30,3),
    'versions':versions,'spider_dev_used_for_training':False,
    'official_test_suite_accuracy':None,'created_utc':datetime.now(timezone.utc).isoformat(),
}
REPORT_PATH=REPORT_DIR/f'09_qlora_7b_{RUN_NAME}_train_report.json'
REPORT_PATH.write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
print('Report:',REPORT_PATH)
print('Training steps finished:',trainer.state.global_step)
print('Validation loss:',validation_metrics.get('eval_loss'))
print('Peak GPU allocated:',report['peak_gpu_memory_allocated_gib'],'GiB')

Report: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results/milestone_B/09_qlora_7b_smoke_10_train_report.json
Training steps finished: 10
Validation loss: 0.16986048221588135
Peak GPU allocated: 10.229 GiB


## 12 — Final QA and run handoff

For the **main** experiment, a successful notebook should show 200 completed steps and an adapter directory at `models/qlora_7b_sft_v1/main_200/final_adapter`.

Once your main run passes, commit this notebook as `notebooks/09_QLoRA_SFT_7B.ipynb`, keeping adapter checkpoints in Drive rather than Git. Notebook 10 will evaluate the 1.5B and 7B adapters using the official Spider test-suite, frozen prompts and consistent decoding parameters.

In [15]:
assert REPORT_PATH.is_file() and (ADAPTER_DIR/'adapter_config.json').is_file()
assert len(train_encoded)>0 and len(val_encoded)>0
assert train_db.isdisjoint(val_db)
assert trainer.state.global_step==MAX_STEPS, (trainer.state.global_step,MAX_STEPS)
assert report['official_test_suite_accuracy'] is None
print('PASS — 7B QLoRA run finished:',RUN_NAME)
print('PASS — Dataset split disjoint; no Spider dev training')
print('PASS — Adapter saved:',ADAPTER_DIR)
print('PASS — Report saved:',REPORT_PATH)
if RUN_NAME=='smoke_10':
    print('NEXT: switch RUN_NAME to main_200 and MAX_STEPS to 200, restart runtime, rerun all sections from 2')
else:
    print('NEXT: Notebook 10 — official comparison against 73.2% frozen 7B baseline')

PASS — 7B QLoRA run finished: smoke_10
PASS — Dataset split disjoint; no Spider dev training
PASS — Adapter saved: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/models/qlora_7b_sft_v1/smoke_10/final_adapter
PASS — Report saved: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results/milestone_B/09_qlora_7b_smoke_10_train_report.json
NEXT: switch RUN_NAME to main_200 and MAX_STEPS to 200, restart runtime, rerun all sections from 2
